# Macro Regime & Market Risk Monitor
## 01 — Data, macro regimes and monetary conditions

Run every cell from top to bottom. This notebook produces the inputs for Notebook 02
and SQL. Its production outputs retain their existing names and columns.

**Scope:** monthly market data from December 2008 through August 2026; completed
quarterly comparisons through 2026Q2. US growth is converted from annualized GDP
growth to quarter-on-quarter growth. Strong growth means > 0.5%; high inflation
means ≥ 3%. Eurozone GDP and unemployment retain EA21 coverage; the final HICP
series retains the source's `EA` aggregate. Series definitions and thresholds are
unchanged.

**Corrections:** US CPI inflation compares the same calendar month one year earlier,
retaining missing months. Yield and sovereign-spread quarterly changes are calculated
on the complete market calendar **before** quarters without a macro regime are removed.
Negative expected real rates describe monetary conditions; Notebook 02 excludes them
from the three-channel market stress counts.

The cached raw snapshot makes repeat runs independent of subsequent data revisions.
Supplementary real-rate, long-end and fiscal analysis appears in the appendix and
does not alter the production regime or stress definitions.

### 1. Paths and snapshot mode

Local use: run from the repository root, or set `MACRO_PROJECT_DIR` to
that folder before running. In Colab, place the project folder (including `data/raw`)
in the runtime or Drive and set that variable to its location. Production CSVs are
written to `OUTPUT_DIR`, normally `data/processed` beneath the repository root.
Set `MACRO_OUTPUT_DIR` to choose a different CSV destination.

The default is **cached** mode. To deliberately retrieve a new vintage, set
`MACRO_DATA_MODE=live`. A FRED key is read from the `FRED_API_KEY` environment variable
or the Colab secret with that name; keys are never printed or saved. A live refresh
can revise past observations, so compare its results before replacing an approved
snapshot. Required packages: pandas, numpy and requests.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import xml.etree.ElementTree as ET
from functools import reduce

import numpy as np
import pandas as pd
import requests
from IPython.display import display

PROJECT_DIR = Path(os.environ.get("MACRO_PROJECT_DIR", str(Path.cwd()))).expanduser().resolve()
RAW_DIR = Path(os.environ.get("MACRO_RAW_DIR", str(PROJECT_DIR / "data" / "raw"))).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("MACRO_OUTPUT_DIR", PROJECT_DIR / "data" / "processed")).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_MODE = os.environ.get("MACRO_DATA_MODE", "cached").lower()
if DATA_MODE not in {"cached", "live"}:
    raise ValueError("MACRO_DATA_MODE must be cached or live.")
ANALYSIS_START = pd.Period("2008-12", freq="M")
ANALYSIS_END = pd.Period("2026-08", freq="M")
QUARTER_END = pd.Period("2026Q2", freq="Q")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
APPENDIX_DIR = PROJECT_DIR / "analysis" / "notebook01"
APPENDIX_DIR.mkdir(parents=True, exist_ok=True)
SNAPSHOT_MANIFEST = RAW_DIR / "manifest.json"
snapshot_manifest = json.loads(SNAPSHOT_MANIFEST.read_text(encoding="utf-8")) if SNAPSHOT_MANIFEST.exists() else {"sources": {}}
snapshot_manifest.setdefault("sources", {})
source_usage = {}
print(f"Data mode: {DATA_MODE}; output folder: {OUTPUT_DIR}")


In [ ]:
def check_key(frame, key, name, complete=False):
    if frame.empty or frame[key].isna().any() or frame[key].duplicated().any():
        raise ValueError(f"{name}: keys must be nonempty, unique and nonmissing.")
    if not frame[key].is_monotonic_increasing:
        raise ValueError(f"{name}: keys must be ordered.")
    if complete:
        expected = pd.period_range(frame[key].min(), frame[key].max(), freq=frame[key].dtype.freq)
        if list(frame[key]) != list(expected):
            raise ValueError(f"{name}: an unexpected calendar gap exists.")

def safe_merge(left, right, key, how="inner"):
    return left.merge(right, on=key, how=how, validate="one_to_one").sort_values(key).reset_index(drop=True)

def read_source(name, provider, identifier, column, fetcher, public_parameters=None):
    path = RAW_DIR / f"{name}.csv"
    if DATA_MODE == "cached":
        if not path.exists():
            raise FileNotFoundError(f"Missing cached source {path.name}. Supply the raw snapshot or deliberately choose live mode.")
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        record = snapshot_manifest["sources"].get(name)
        if record is None or record.get("sha256") != digest:
            raise ValueError(f"Snapshot manifest missing or hash mismatch for {name}.")
        frame = pd.read_csv(path)
        if list(frame.columns) != ["date", "value"]:
            raise ValueError(f"{name}: cached observations require date and value columns.")
        frame = frame.rename(columns={"value": column})
    else:
        frame = fetcher()
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        frame.rename(columns={column: "value"}).to_csv(path, index=False)
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        snapshot_manifest["sources"][name] = {
            "provider": provider, "identifier": identifier,
            "parameters": public_parameters or {}, "file": path.name,
            "sha256": digest, "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        SNAPSHOT_MANIFEST.write_text(json.dumps(snapshot_manifest, indent=2), encoding="utf-8")
    if list(frame.columns) != ["date", column]:
        raise ValueError(f"{name}: unexpected source columns.")
    frame["date"] = pd.to_datetime(frame["date"], errors="raise")
    frame[column] = pd.to_numeric(frame[column], errors="raise")
    frame = frame.sort_values("date").reset_index(drop=True)
    check_key(frame, "date", name)
    if np.isinf(frame[column].to_numpy(dtype=float)).any():
        raise ValueError(f"{name}: infinite source value.")
    source_usage[name] = {"file": path.name, "sha256": digest, "rows": len(frame)}
    return frame

def get_fred_key():
    key = os.environ.get("FRED_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
        key = userdata.get("FRED_API_KEY")
    except Exception:
        key = None
    if not key:
        raise RuntimeError("Live FRED retrieval requires FRED_API_KEY in the environment or Colab secrets.")
    return key

def safe_request(url, params=None):
    try:
        response = requests.get(url, params=params, timeout=(15, 90))
    except requests.RequestException:
        raise RuntimeError("Data provider request failed; check connectivity and retry.") from None
    if not response.ok:
        # Avoid an exception containing a URL with the private FRED API key.
        raise RuntimeError(f"Data provider returned HTTP {response.status_code}.")
    return response

def get_fred_series(series_id, column_name):
    def fetch():
        response = safe_request("https://api.stlouisfed.org/fred/series/observations", {
            "series_id": series_id, "api_key": get_fred_key(), "file_type": "json"
        })
        payload = response.json()
        if "observations" not in payload:
            raise RuntimeError(f"FRED returned no observations for {series_id}.")
        rows = pd.DataFrame(payload["observations"])[["date", "value"]]
        rows["value"] = pd.to_numeric(rows["value"].replace(".", np.nan), errors="raise")
        return rows.rename(columns={"value": column_name})
    return read_source(f"fred_{series_id}", "FRED", series_id, column_name, fetch)

def get_ecb_series(series_key, column_name, cache_name):
    def fetch():
        response = safe_request(f"https://data-api.ecb.europa.eu/service/data/{series_key}")
        root = ET.fromstring(response.text)
        namespace = {"generic": "http://www.sdmx.org/resources/sdmxml/schemas/v2_1/data/generic"}
        rows = []
        for observation in root.findall(".//generic:Obs", namespace):
            date_node = observation.find("generic:ObsDimension", namespace)
            value_node = observation.find("generic:ObsValue", namespace)
            if date_node is not None:
                rows.append([date_node.attrib["value"], value_node.attrib["value"] if value_node is not None else np.nan])
        if not rows:
            raise RuntimeError(f"ECB returned no observations for {cache_name}.")
        return pd.DataFrame(rows, columns=["date", column_name])
    return read_source(cache_name, "ECB", series_key, column_name, fetch)

def get_eurostat_series(dataset, parameters, column_name, cache_name):
    def fetch():
        response = safe_request(f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/{dataset}", parameters)
        payload = response.json()
        ids, sizes = payload["id"], payload["size"]
        if "time" not in ids or any(size != 1 for name, size in zip(ids, sizes) if name != "time"):
            raise ValueError(f"{cache_name}: API parameters must select a single series.")
        index = payload["dimension"]["time"]["category"]["index"]
        if isinstance(index, list):
            index = {period: position for position, period in enumerate(index)}
        values = payload.get("value", {})
        rows = []
        for period, position in sorted(index.items(), key=lambda pair: pair[1]):
            value = values[position] if isinstance(values, list) else values.get(str(position), np.nan)
            if "Q" in period:
                date = pd.Period(period, freq="Q").start_time
            else:
                date = pd.Period(period, freq="M").start_time
            rows.append([date, value])
        return pd.DataFrame(rows, columns=["date", column_name])
    return read_source(cache_name, "Eurostat", dataset, column_name, fetch, parameters)

def month_end(frame):
    frame = frame.sort_values("date").set_index("date").resample("ME").last().reset_index()
    frame["month"] = frame["date"].dt.to_period("M")
    return frame

def curve_label(values):
    if values.isna().any():
        raise ValueError("A missing yield spread cannot receive a curve label.")
    return np.select([values < 0, values < 0.25], ["Inverted", "Flat"], default="Positive")

def quarterly_last(frame, columns):
    data = frame[["month", *columns]].sort_values("month").copy()
    data["quarter"] = data["month"].dt.asfreq("Q")
    result = data.groupby("quarter", sort=True)[columns].last().reset_index()
    full_calendar = pd.period_range(result["quarter"].min(), result["quarter"].max(), freq="Q")
    return result.set_index("quarter").reindex(full_calendar).rename_axis("quarter").reset_index()

def quarterly_complete_mean(frame, value, name=None):
    data = frame.copy()
    check_key(data.sort_values("month"), "month", f"{value} monthly values")
    data["quarter"] = data["month"].dt.asfreq("Q")
    result = data.groupby("quarter", sort=True).agg(mean=(value, "mean"), months_available=(value, "count")).reset_index()
    incomplete = result.loc[result["months_available"] != 3, ["quarter", "months_available"]].copy()
    result = result.loc[result["months_available"] == 3, ["quarter", "mean"]].rename(columns={"mean": name or value})
    return result, incomplete


### 2. US observations and exact calendar inflation

FRED yields and target-rate bounds remain daily series; monthly values remain the
last available observation. Reindexing CPI creates an explicit row for each calendar
month. The 12-month comparison uses `fill_method=None`, so a missing month stays
missing instead of turning the comparison into a 13-month interval.

In [ ]:
us_unemployment = get_fred_series("UNRATE", "unemployment_rate")
us_cpi = get_fred_series("CPIAUCSL", "cpi")
us_2y = get_fred_series("DGS2", "us_2y_yield")
us_10y = get_fred_series("DGS10", "us_10y_yield")
us_policy_lower = get_fred_series("DFEDTARL", "policy_rate_lower")
us_policy_upper = get_fred_series("DFEDTARU", "policy_rate_upper")
us_gdp = get_fred_series("A191RL1Q225SBEA", "real_gdp_growth")

us_cpi["month"] = us_cpi["date"].dt.to_period("M")
check_key(us_cpi.sort_values("month"), "month", "US CPI")
cpi_calendar = pd.period_range(us_cpi["month"].min(), us_cpi["month"].max(), freq="M")
us_cpi = us_cpi.set_index("month")[['cpi']].reindex(cpi_calendar).rename_axis("month").reset_index()
us_cpi["date"] = us_cpi["month"].dt.to_timestamp()
us_cpi["us_cpi_yoy"] = us_cpi["cpi"].pct_change(periods=12, fill_method=None) * 100
check_key(us_cpi, "month", "US CPI calendar", complete=True)

us_yield_curve = safe_merge(us_2y.dropna(), us_10y.dropna(), "date")
us_yield_curve["us_10y_2y_spread"] = us_yield_curve["us_10y_yield"] - us_yield_curve["us_2y_yield"]
us_yield_curve_monthly = month_end(us_yield_curve)
us_policy = safe_merge(us_policy_lower.dropna(), us_policy_upper.dropna(), "date")
us_policy["us_policy_rate"] = (us_policy["policy_rate_lower"] + us_policy["policy_rate_upper"]) / 2
us_policy_monthly = month_end(us_policy)[["month", "us_policy_rate"]]
us_cpi_monthly = us_cpi[["month", "us_cpi_yoy"]].copy()
us_gdp_quarterly = us_gdp.dropna().copy()
us_gdp_quarterly["quarter"] = us_gdp_quarterly["date"].dt.to_period("Q")
us_gdp_quarterly["real_gdp_growth_qoq"] = ((1 + us_gdp_quarterly["real_gdp_growth"] / 100) ** (1 / 4) - 1) * 100
check_key(us_gdp_quarterly, "quarter", "US GDP")


### 3. Eurozone observations

The Eurostat queries select one series explicitly. Missing values remain missing.
The ECB yield curve retains the fitted Eurozone 2Y and 10Y series; Germany and Italy
retain their separate monthly government-bond yield series.

In [ ]:
eurozone_gdp = get_eurostat_series("namq_10_gdp", {
    "format": "JSON", "lang": "en", "freq": "Q", "unit": "CLV_PCH_PRE",
    "na_item": "B1GQ", "geo": "EA21", "s_adj": "SCA"
}, "real_gdp_growth", "eurostat_gdp_ea21")
eurozone_gdp["quarter"] = eurozone_gdp["date"].dt.to_period("Q")
eurozone_gdp["real_gdp_growth_qoq"] = eurozone_gdp["real_gdp_growth"]
eurozone_hicp = get_eurostat_series("prc_hicp_minr", {
    "format": "JSON", "lang": "en", "freq": "M", "unit": "RCH_A",
    "coicop18": "TOTAL", "geo": "EA"
}, "hicp_yoy", "eurostat_hicp_ea")
eurozone_hicp["month"] = eurozone_hicp["date"].dt.to_period("M")
eurozone_hicp = eurozone_hicp[["month", "hicp_yoy"]]
eurozone_unemployment = get_eurostat_series("une_rt_m", {
    "format": "JSON", "lang": "en", "freq": "M", "s_adj": "SA",
    "age": "TOTAL", "unit": "PC_ACT", "sex": "T", "geo": "EA21"
}, "unemployment_rate", "eurostat_unemployment_ea21")
eurozone_unemployment["month"] = eurozone_unemployment["date"].dt.to_period("M")
eurozone_unemployment = eurozone_unemployment[["month", "unemployment_rate"]]

eurozone_2y = get_ecb_series("YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y", "eurozone_2y_yield", "ecb_eurozone_2y")
eurozone_10y = get_ecb_series("YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y", "eurozone_10y_yield", "ecb_eurozone_10y")
germany_10y = get_ecb_series("IRS/M.DE.L.L40.CI.0000.EUR.N.Z", "germany_10y_yield", "ecb_germany_10y")
italy_10y = get_ecb_series("IRS/M.IT.L.L40.CI.0000.EUR.N.Z", "italy_10y_yield", "ecb_italy_10y")
eurozone_policy = get_ecb_series("FM/D.U2.EUR.4F.KR.DFR.LEV", "eurozone_policy_rate", "ecb_policy")

eurozone_yield_curve = safe_merge(eurozone_2y.dropna(), eurozone_10y.dropna(), "date")
eurozone_yield_curve["eurozone_10y_2y_spread"] = eurozone_yield_curve["eurozone_10y_yield"] - eurozone_yield_curve["eurozone_2y_yield"]
eurozone_yield_curve_monthly = month_end(eurozone_yield_curve)
eurozone_policy_monthly = month_end(eurozone_policy)[["month", "eurozone_policy_rate"]]
sovereign_spread_analysis = safe_merge(germany_10y, italy_10y, "date")
sovereign_spread_analysis["month"] = sovereign_spread_analysis["date"].dt.to_period("M")
sovereign_spread_analysis["italy_germany_10y_spread"] = sovereign_spread_analysis["italy_10y_yield"] - sovereign_spread_analysis["germany_10y_yield"]
check_key(sovereign_spread_analysis, "month", "Italy–Germany monthly spread")
sovereign_spread_analysis["spread_change"] = sovereign_spread_analysis["italy_germany_10y_spread"].diff()


### 4. Macro classification

Quarterly inflation requires all three monthly values. No missing GDP or inflation
value is assigned a regime. The regional comparison contains only quarters where
both regions can be classified. A coverage table records excluded quarters.

In [ ]:
us_cpi_quarterly, us_cpi_incomplete = quarterly_complete_mean(us_cpi_monthly, "us_cpi_yoy")
eurozone_hicp_quarterly, eurozone_hicp_incomplete = quarterly_complete_mean(eurozone_hicp, "hicp_yoy")

def classify_macro(gdp, inflation, inflation_column):
    result = safe_merge(gdp[["quarter", "real_gdp_growth_qoq"]], inflation, "quarter")
    result = result.dropna(subset=["real_gdp_growth_qoq", inflation_column]).copy()
    result = result[result["quarter"] <= QUARTER_END].reset_index(drop=True)
    result["growth_regime"] = np.where(result["real_gdp_growth_qoq"] > 0.5, "Strong Growth", "Weak Growth")
    result["inflation_regime"] = np.where(result[inflation_column] >= 3.0, "High Inflation", "Low Inflation")
    result["regime"] = np.select([
        (result["growth_regime"] == "Strong Growth") & (result["inflation_regime"] == "Low Inflation"),
        (result["growth_regime"] == "Strong Growth") & (result["inflation_regime"] == "High Inflation"),
        (result["growth_regime"] == "Weak Growth") & (result["inflation_regime"] == "High Inflation"),
    ], ["Goldilocks", "Overheating", "Stagflation"], default="Deflationary Slowdown")
    return result

us_regime = classify_macro(us_gdp_quarterly, us_cpi_quarterly, "us_cpi_yoy")
eurozone_regime = classify_macro(eurozone_gdp, eurozone_hicp_quarterly, "hicp_yoy")
regional_regime = safe_merge(
    us_regime[["quarter", "regime"]].rename(columns={"regime": "us_regime"}),
    eurozone_regime[["quarter", "regime"]].rename(columns={"regime": "eurozone_regime"}), "quarter"
)
regional_regime["regime_divergence"] = regional_regime["us_regime"] != regional_regime["eurozone_regime"]
macro_regime_signals = regional_regime.copy()
check_key(regional_regime, "quarter", "Regional regimes")
display(regional_regime.tail(8))


### 5. Monthly market conditions

Policy and realized real-rate proxies retain nominal rate minus observed annual
inflation. October 2025 US inflation and its two realized real-rate proxies remain
blank. Yield-curve labels retain Inverted < 0, Flat from 0 to < 0.25, and Positive
≥ 0.25. The final curve series retains the original common US/ECB daily calendar.

In [ ]:
us_policy_signals = safe_merge(us_policy_monthly, us_yield_curve_monthly[["month", "us_2y_yield"]], "month", "outer")
us_policy_signals = safe_merge(us_policy_signals, us_cpi_monthly, "month", "outer")
us_policy_signals["us_real_policy_rate"] = us_policy_signals["us_policy_rate"] - us_policy_signals["us_cpi_yoy"]
us_policy_signals["us_real_2y_rate"] = us_policy_signals["us_2y_yield"] - us_policy_signals["us_cpi_yoy"]
us_policy_signals["us_2y_policy_gap"] = us_policy_signals["us_2y_yield"] - us_policy_signals["us_policy_rate"]

eurozone_policy_signals = safe_merge(eurozone_policy_monthly, eurozone_yield_curve_monthly[["month", "eurozone_2y_yield"]], "month", "outer")
eurozone_policy_signals = safe_merge(eurozone_policy_signals, eurozone_hicp, "month", "outer")
eurozone_policy_signals["eurozone_real_policy_rate"] = eurozone_policy_signals["eurozone_policy_rate"] - eurozone_policy_signals["hicp_yoy"]
eurozone_policy_signals["eurozone_real_2y_rate"] = eurozone_policy_signals["eurozone_2y_yield"] - eurozone_policy_signals["hicp_yoy"]
eurozone_policy_signals["eurozone_2y_policy_gap"] = eurozone_policy_signals["eurozone_2y_yield"] - eurozone_policy_signals["eurozone_policy_rate"]

POLICY_COLUMNS = ["month", "us_policy_rate", "us_cpi_yoy", "us_real_policy_rate", "us_2y_yield", "us_real_2y_rate", "us_2y_policy_gap", "eurozone_policy_rate", "hicp_yoy", "eurozone_real_policy_rate", "eurozone_2y_yield", "eurozone_real_2y_rate", "eurozone_2y_policy_gap"]
policy_signals = safe_merge(us_policy_signals, eurozone_policy_signals, "month", "outer")[POLICY_COLUMNS]
policy_stance = policy_signals.copy()

common_daily_curve = safe_merge(us_yield_curve, eurozone_yield_curve, "date")
yield_curve_monthly = month_end(common_daily_curve)
within_scope = yield_curve_monthly["month"].between(ANALYSIS_START, ANALYSIS_END)
yield_curve_monthly = yield_curve_monthly.loc[within_scope].reset_index(drop=True)
yield_curve_monthly["us_curve_regime"] = curve_label(yield_curve_monthly["us_10y_2y_spread"])
yield_curve_monthly["eurozone_curve_regime"] = curve_label(yield_curve_monthly["eurozone_10y_2y_spread"])
CURVE_COLUMNS = ["month", "us_10y_yield", "us_10y_2y_spread", "us_curve_regime", "eurozone_10y_yield", "eurozone_10y_2y_spread", "eurozone_curve_regime"]
curve_signals = yield_curve_monthly[CURVE_COLUMNS].copy()
SOVEREIGN_COLUMNS = ["month", "germany_10y_yield", "italy_10y_yield", "italy_germany_10y_spread", "spread_change"]
sovereign_signals = sovereign_spread_analysis[SOVEREIGN_COLUMNS].copy()
policy_signals = policy_signals.loc[policy_signals["month"].between(ANALYSIS_START, ANALYSIS_END)].reset_index(drop=True)
sovereign_signals = sovereign_signals.loc[sovereign_signals["month"].between(ANALYSIS_START, ANALYSIS_END)].reset_index(drop=True)
monthly_market_signals = safe_merge(safe_merge(policy_signals, curve_signals, "month"), sovereign_signals, "month")
MONTHLY_COLUMNS = POLICY_COLUMNS + CURVE_COLUMNS[1:] + SOVEREIGN_COLUMNS[1:]
monthly_market_signals = monthly_market_signals[MONTHLY_COLUMNS]
check_key(monthly_market_signals, "month", "Monthly market signals", complete=True)
if list(monthly_market_signals["month"]) != list(pd.period_range(ANALYSIS_START, ANALYSIS_END, freq="M")):
    raise ValueError("Monthly source coverage does not fill the approved analysis window.")


### 6. Expected real rates and the complete market-quarter calendar

The existing expected real-rate proxies are retained: quarter-end 2Y yield minus
the quarterly mean of US expected inflation or the first Eurozone SPF observation
in that quarter. Selecting the first SPF observation is explicit and recorded.

The first market quarter, 2008Q4, retains blank QoQ changes because it is the start
of this analysis window. Later changes compare adjacent **calendar quarters**, even
when a quarter is absent from the macro regime table. All three differences are
calculated here; no `.diff()` is applied after the macro join.

In [ ]:
us_expected_inflation = get_fred_series("EXPINF2YR", "us_expected_inflation_2y")
us_expected_inflation["month"] = us_expected_inflation["date"].dt.to_period("M")
us_expected_inflation_quarterly, expected_inflation_incomplete = quarterly_complete_mean(us_expected_inflation[["month", "us_expected_inflation_2y"]], "us_expected_inflation_2y")
eurozone_expected_inflation = get_ecb_series("SPF/M.U2.HICP.POINT.P24M.Q.AVG", "eurozone_expected_inflation_2y", "ecb_expected_inflation")
eurozone_expected_inflation["quarter"] = eurozone_expected_inflation["date"].dt.to_period("Q")
spf_multiple_observations = eurozone_expected_inflation.groupby("quarter").size().rename("observations").reset_index()
spf_multiple_observations = spf_multiple_observations[spf_multiple_observations["observations"] > 1]
eurozone_expected_inflation_quarterly = eurozone_expected_inflation.sort_values("date").drop_duplicates("quarter", keep="first")[["quarter", "eurozone_expected_inflation_2y"]]

us_2y_quarterly = quarterly_last(us_yield_curve_monthly, ["us_2y_yield"])
eurozone_2y_quarterly = quarterly_last(eurozone_yield_curve_monthly, ["eurozone_2y_yield"])
forward_real_rate = safe_merge(us_2y_quarterly, us_expected_inflation_quarterly, "quarter")
forward_real_rate = safe_merge(forward_real_rate, eurozone_2y_quarterly, "quarter")
forward_real_rate = safe_merge(forward_real_rate, eurozone_expected_inflation_quarterly, "quarter")
forward_real_rate = forward_real_rate[forward_real_rate["quarter"] <= QUARTER_END].reset_index(drop=True)
forward_real_rate["us_expected_real_2y_rate"] = forward_real_rate["us_2y_yield"] - forward_real_rate["us_expected_inflation_2y"]
forward_real_rate["eurozone_expected_real_2y_rate"] = forward_real_rate["eurozone_2y_yield"] - forward_real_rate["eurozone_expected_inflation_2y"]

MARKET_QUARTER_COLUMNS = ["us_10y_2y_spread", "us_curve_regime", "eurozone_10y_2y_spread", "eurozone_curve_regime", "italy_germany_10y_spread", "us_10y_yield", "eurozone_10y_yield"]
quarterly_market_full = quarterly_last(monthly_market_signals, MARKET_QUARTER_COLUMNS)
quarterly_market_full = quarterly_market_full[quarterly_market_full["quarter"] <= QUARTER_END].reset_index(drop=True)
check_key(quarterly_market_full, "quarter", "Full market quarters", complete=True)
core_market = [column for column in MARKET_QUARTER_COLUMNS if not column.endswith("regime")]
if quarterly_market_full[core_market].isna().any().any():
    raise ValueError("A missing market quarter cannot be bridged by a QoQ calculation.")
quarterly_market_full["italy_germany_spread_qoq_change"] = quarterly_market_full["italy_germany_10y_spread"].diff()
quarterly_market_full["us_10y_qoq_change"] = quarterly_market_full["us_10y_yield"].diff()
quarterly_market_full["eurozone_10y_qoq_change"] = quarterly_market_full["eurozone_10y_yield"].diff()


### 7. Regime and market confirmation

This join retains the existing market-confirmation output contract. Curve and
fragmentation labels keep their original thresholds. The combined forward-rate
label remains a monetary-condition indicator. The older broad-confirmation label
is retained for the supplementary analysis; Notebook 02's production stress score
uses the corrected three market channels.

In [ ]:
market_confirmation = safe_merge(regional_regime, quarterly_market_full, "quarter")
market_confirmation = safe_merge(market_confirmation, forward_real_rate[["quarter", "us_expected_real_2y_rate", "eurozone_expected_real_2y_rate"]], "quarter")
market_confirmation["curve_confirmation"] = (
    (market_confirmation["us_regime"].isin(["Stagflation", "Deflationary Slowdown"]) | market_confirmation["eurozone_regime"].isin(["Stagflation", "Deflationary Slowdown"])) &
    (market_confirmation["us_curve_regime"].isin(["Inverted", "Flat"]) | market_confirmation["eurozone_curve_regime"].isin(["Inverted", "Flat"]))
)
us_curve_stress = market_confirmation["us_curve_regime"].isin(["Inverted", "Flat"])
eurozone_curve_stress = market_confirmation["eurozone_curve_regime"].isin(["Inverted", "Flat"])
market_confirmation["curve_signal"] = np.select([us_curve_stress & eurozone_curve_stress, us_curve_stress | eurozone_curve_stress], ["Both Curves Stress", "One Curve Stress"], default="No Curve Stress")
us_negative = market_confirmation["us_expected_real_2y_rate"] < 0
eurozone_negative = market_confirmation["eurozone_expected_real_2y_rate"] < 0
market_confirmation["forward_real_rate_signal"] = np.select([us_negative & eurozone_negative, us_negative | eurozone_negative], ["Both Negative", "One Negative"], default="Both Positive")
market_confirmation["fragmentation_signal"] = np.select([
    market_confirmation["italy_germany_spread_qoq_change"] > 0,
    market_confirmation["italy_germany_spread_qoq_change"] < 0
], ["Widening", "Narrowing"], default="Unchanged")
market_confirmation["market_confirmation_signal"] = np.select([
    (market_confirmation["curve_signal"] == "Both Curves Stress") & (market_confirmation["forward_real_rate_signal"] == "Both Negative") & (market_confirmation["fragmentation_signal"] == "Widening"),
    (market_confirmation["curve_signal"] == "No Curve Stress") & (market_confirmation["forward_real_rate_signal"] == "Both Positive") & (market_confirmation["fragmentation_signal"] == "Narrowing")
], ["Broad Market Stress", "Broad Market Confirmation"], default="Mixed Market Signals")
for region in ["us", "eurozone"]:
    market_confirmation[f"{region}_10y_direction"] = np.select([
        market_confirmation[f"{region}_10y_qoq_change"] > 0,
        market_confirmation[f"{region}_10y_qoq_change"] < 0
    ], ["Rising", "Falling"], default="Unchanged")

us_10y_breakeven = get_fred_series("T10YIE", "us_10y_breakeven")
us_10y_breakeven["quarter"] = us_10y_breakeven["date"].dt.to_period("Q")
us_10y_breakeven_quarterly = us_10y_breakeven.dropna().sort_values("date").groupby("quarter").last().reset_index()[["quarter", "us_10y_breakeven"]]
market_confirmation = safe_merge(market_confirmation, us_10y_breakeven_quarterly, "quarter", "left")
market_confirmation["us_10y_real_yield_proxy"] = market_confirmation["us_10y_yield"] - market_confirmation["us_10y_breakeven"]
MARKET_CONFIRMATION_COLUMNS = ["quarter", "us_regime", "eurozone_regime", "regime_divergence", "us_10y_2y_spread", "us_curve_regime", "eurozone_10y_2y_spread", "eurozone_curve_regime", "italy_germany_10y_spread", "us_expected_real_2y_rate", "eurozone_expected_real_2y_rate", "italy_germany_spread_qoq_change", "curve_confirmation", "curve_signal", "forward_real_rate_signal", "fragmentation_signal", "market_confirmation_signal", "us_10y_yield", "eurozone_10y_yield", "us_10y_qoq_change", "eurozone_10y_qoq_change", "us_10y_direction", "eurozone_10y_direction", "us_10y_breakeven", "us_10y_real_yield_proxy"]
market_confirmation = market_confirmation[MARKET_CONFIRMATION_COLUMNS]
display(market_confirmation.tail(4).T)


### 8. Coverage and export validation

Missing October 2025 CPI is retained, with no interpolation. Quarterly classification
requires three valid inflation months, so 2025Q4 is omitted from the regional regime
and cross-asset tables. July and August 2026 monthly observations remain available,
while completed-quarter exports stop at 2026Q2. The coverage CSVs make these limits
explicit rather than silently filling them.

In [ ]:
checks = []
def record_check(name, condition):
    if not bool(condition):
        raise AssertionError(f"Validation failed: {name}")
    checks.append({"check_name": name, "status": "passed"})

record_check("macro regime schema", list(macro_regime_signals.columns) == ["quarter", "us_regime", "eurozone_regime", "regime_divergence"])
record_check("monthly market schema", list(monthly_market_signals.columns) == MONTHLY_COLUMNS and len(MONTHLY_COLUMNS) == 23)
record_check("market confirmation schema", list(market_confirmation.columns) == MARKET_CONFIRMATION_COLUMNS)
for frame, key, name in [(macro_regime_signals, "quarter", "macro regimes"), (monthly_market_signals, "month", "monthly markets"), (market_confirmation, "quarter", "market confirmation")]:
    check_key(frame, key, name)
    record_check(f"{name} keys", frame[key].is_unique and frame[key].is_monotonic_increasing)
    numeric = frame.select_dtypes(include="number").to_numpy(dtype=float)
    record_check(f"{name} finite values", not np.isinf(numeric).any())
record_check("regime divergence identity", (macro_regime_signals["regime_divergence"] == (macro_regime_signals["us_regime"] != macro_regime_signals["eurozone_regime"])).all())
record_check("monthly analysis calendar", len(monthly_market_signals) == len(pd.period_range(ANALYSIS_START, ANALYSIS_END, freq="M")))
record_check("quarterly cutoff", market_confirmation["quarter"].max() <= QUARTER_END and macro_regime_signals["quarter"].max() <= QUARTER_END)
record_check("expected real rates present", not market_confirmation[["us_expected_real_2y_rate", "eurozone_expected_real_2y_rate"]].isna().any().any())
monthly_missing = monthly_market_signals.isna()
allowed_missing = pd.DataFrame(False, index=monthly_market_signals.index, columns=monthly_market_signals.columns)
allowed_missing.loc[monthly_market_signals["month"] == pd.Period("2025-10", freq="M"), ["us_cpi_yoy", "us_real_policy_rate", "us_real_2y_rate"]] = True
record_check("monthly missing values limited to known CPI gap", not (monthly_missing & ~allowed_missing).any().any())
first_market_quarter = quarterly_market_full["quarter"].min()
for column in ["us_10y_qoq_change", "eurozone_10y_qoq_change", "italy_germany_spread_qoq_change"]:
    record_check(f"{column} null only at first market quarter", (market_confirmation[column].isna() == (market_confirmation["quarter"] == first_market_quarter)).all())
required_market_columns = [column for column in MARKET_CONFIRMATION_COLUMNS if column not in {"us_10y_qoq_change", "eurozone_10y_qoq_change", "italy_germany_spread_qoq_change"}]
record_check("market confirmation required values present", not market_confirmation[required_market_columns].isna().any().any())
record_check("production regime values present", not macro_regime_signals.isna().any().any())
comparison = safe_merge(market_confirmation[["quarter", "us_10y_qoq_change", "eurozone_10y_qoq_change", "italy_germany_spread_qoq_change"]], quarterly_market_full[["quarter", "us_10y_qoq_change", "eurozone_10y_qoq_change", "italy_germany_spread_qoq_change"]].rename(columns={column: f"{column}_calendar" for column in ["us_10y_qoq_change", "eurozone_10y_qoq_change", "italy_germany_spread_qoq_change"]}), "quarter")
for column in ["us_10y_qoq_change", "eurozone_10y_qoq_change", "italy_germany_spread_qoq_change"]:
    record_check(f"{column} survives macro join", np.allclose(comparison[column], comparison[f"{column}_calendar"], equal_nan=True, atol=1e-12, rtol=0))
baseline = us_cpi.set_index("month")["cpi"]
for month in us_cpi.loc[us_cpi["us_cpi_yoy"].notna(), "month"]:
    previous = month - 12
    expected = (baseline.loc[month] / baseline.loc[previous] - 1) * 100
    if not np.isclose(us_cpi.loc[us_cpi["month"] == month, "us_cpi_yoy"].iloc[0], expected, atol=1e-12, rtol=0):
        raise AssertionError("US inflation is not aligned to the same calendar month.")
record_check("US inflation exact calendar year", True)

regime_calendar = pd.DataFrame({"quarter": pd.period_range(regional_regime["quarter"].min(), QUARTER_END, freq="Q")})
regime_coverage = safe_merge(regime_calendar, us_gdp_quarterly[["quarter", "real_gdp_growth_qoq"]].rename(columns={"real_gdp_growth_qoq": "us_growth_qoq"}), "quarter", "left")
regime_coverage = safe_merge(regime_coverage, us_cpi_monthly.assign(quarter=us_cpi_monthly["month"].dt.asfreq("Q")).groupby("quarter")["us_cpi_yoy"].count().rename("us_inflation_months").reset_index(), "quarter", "left")
regime_coverage = safe_merge(regime_coverage, eurozone_gdp[["quarter", "real_gdp_growth_qoq"]].rename(columns={"real_gdp_growth_qoq": "eurozone_growth_qoq"}), "quarter", "left")
regime_coverage = safe_merge(regime_coverage, eurozone_hicp.assign(quarter=eurozone_hicp["month"].dt.asfreq("Q")).groupby("quarter")["hicp_yoy"].count().rename("eurozone_inflation_months").reset_index(), "quarter", "left")
regime_coverage["included_in_regional_regime"] = regime_coverage["quarter"].isin(regional_regime["quarter"])
regime_coverage["included_in_market_confirmation"] = regime_coverage["quarter"].isin(market_confirmation["quarter"])
missing_monthly_values = monthly_market_signals.set_index("month").isna().stack()
missing_monthly_values = missing_monthly_values[missing_monthly_values].rename("missing").reset_index().rename(columns={"level_1": "column"})
print(f"Validation passed: {len(checks)} checks; {len(macro_regime_signals)} regime quarters; {len(monthly_market_signals)} months; {len(market_confirmation)} market quarters.")
display(regime_coverage.loc[~regime_coverage["included_in_regional_regime"]])
display(missing_monthly_values)


In [ ]:
production_outputs = {
    "macro_regime_signals.csv": macro_regime_signals,
    "monthly_market_signals.csv": monthly_market_signals,
    "regional_regime.csv": regional_regime,
    "us_regime.csv": us_regime,
    "market_confirmation.csv": market_confirmation,
}
for filename, frame in production_outputs.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False, lineterminator="\r\n")
    reloaded = pd.read_csv(OUTPUT_DIR / filename)
    record_check(f"{filename} export shape", list(reloaded.columns) == list(frame.columns) and len(reloaded) == len(frame))
pd.DataFrame(checks).to_csv(APPENDIX_DIR / "validation_results.csv", index=False)
regime_coverage.to_csv(APPENDIX_DIR / "regime_coverage.csv", index=False)
missing_monthly_values.to_csv(APPENDIX_DIR / "missing_monthly_values.csv", index=False)
quarterly_market_full.to_csv(APPENDIX_DIR / "quarterly_market_full.csv", index=False)
spf_multiple_observations.to_csv(APPENDIX_DIR / "spf_multiple_observations.csv", index=False)
run_metadata = {
    "run_at_utc": datetime.now(timezone.utc).isoformat(), "data_mode": DATA_MODE,
    "analysis_start": str(ANALYSIS_START), "analysis_end": str(ANALYSIS_END), "quarter_end": str(QUARTER_END),
    "pandas_version": pd.__version__, "numpy_version": np.__version__,
    "raw_sources_used": source_usage,
    "exports": {filename: {"rows": len(frame), "columns": list(frame.columns), "sha256": hashlib.sha256((OUTPUT_DIR / filename).read_bytes()).hexdigest()} for filename, frame in production_outputs.items()},
}
(APPENDIX_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
print("Production exports validated and saved. Continue with Notebook 02.")


## Appendix — supplementary analysis

These tables retain the original exploratory financial work and are saved separately
from the SQL inputs. Historical cross-tabs and correlations are descriptive; they
are not causal evidence or trading rules. The older broad-market confirmation label
and observed-inflation real-rate proxies are retained here for context, without
contributing to Notebook 02's production stress counts.

In [ ]:
divergence_matrix = pd.crosstab(regional_regime["us_regime"], regional_regime["eurozone_regime"])
us_10y_direction_pct = pd.crosstab(market_confirmation["us_regime"], market_confirmation["us_10y_direction"], normalize="index").mul(100).round(1)
eurozone_10y_direction_pct = pd.crosstab(market_confirmation["eurozone_regime"], market_confirmation["eurozone_10y_direction"], normalize="index").mul(100).round(1)
divergence_matrix.to_csv(APPENDIX_DIR / "regime_divergence_matrix.csv")
us_10y_direction_pct.to_csv(APPENDIX_DIR / "us_yield_direction_by_regime.csv")
eurozone_10y_direction_pct.to_csv(APPENDIX_DIR / "eurozone_yield_direction_by_regime.csv")
display(divergence_matrix)


### A. Expected versus observed-inflation real-rate proxies

Quarter-end observed-inflation proxies retain the last available monthly value in
the quarter, matching the original exploratory calculation. Sign disagreements
show how the inflation measure affects the interpretation of monetary conditions.
Zero is treated as nonnegative in both comparisons.

In [ ]:
realized_quarterly = quarterly_last(monthly_market_signals, ["us_real_2y_rate", "eurozone_real_2y_rate"])
real_rate_comparison = safe_merge(realized_quarterly, forward_real_rate[["quarter", "us_expected_real_2y_rate", "eurozone_expected_real_2y_rate"]], "quarter")
real_rate_comparison = real_rate_comparison[real_rate_comparison["quarter"] <= QUARTER_END].copy()
for region in ["us", "eurozone"]:
    real_rate_comparison[f"{region}_forward_vs_expost_gap"] = real_rate_comparison[f"{region}_expected_real_2y_rate"] - real_rate_comparison[f"{region}_real_2y_rate"]
    real_rate_comparison[f"{region}_expost_negative"] = real_rate_comparison[f"{region}_real_2y_rate"] < 0
    real_rate_comparison[f"{region}_forward_negative"] = real_rate_comparison[f"{region}_expected_real_2y_rate"] < 0
    real_rate_comparison[f"{region}_real_rate_disagreement"] = real_rate_comparison[f"{region}_expost_negative"] != real_rate_comparison[f"{region}_forward_negative"]
real_rate_summary = pd.DataFrame({region: [real_rate_comparison[f"{prefix}_real_rate_disagreement"].sum(), len(real_rate_comparison), real_rate_comparison[f"{prefix}_real_rate_disagreement"].mean() * 100] for region, prefix in [("US", "us"), ("Eurozone", "eurozone")]}, index=["Disagreement quarters", "Total quarters", "Disagreement rate (%)"])
real_rate_comparison.to_csv(APPENDIX_DIR / "real_rate_comparison.csv", index=False)
real_rate_summary.to_csv(APPENDIX_DIR / "real_rate_summary.csv")
display(real_rate_summary)


### B. Long-end yield decomposition

The US long-end real-yield proxy remains 10Y nominal yield minus 10Y breakeven
inflation. It is a separate descriptive proxy; it is not used to calculate the
regional stress counts or the expected 2Y real-rate indicators.

In [ ]:
long_end_comparison = market_confirmation[["quarter", "us_regime", "us_10y_yield", "us_10y_qoq_change", "us_10y_breakeven", "us_10y_real_yield_proxy", "eurozone_regime", "eurozone_10y_yield", "eurozone_10y_qoq_change"]].copy()
long_end_comparison["us_10y_signal"] = market_confirmation["us_10y_direction"]
long_end_comparison["eurozone_10y_signal"] = market_confirmation["eurozone_10y_direction"]
long_end_comparison.to_csv(APPENDIX_DIR / "long_end_comparison.csv", index=False)
display(long_end_comparison.tail(4))


### C. Fiscal balance and yield comparisons

US net lending/borrowing divided by nominal GDP and the Eurostat fiscal-balance
series retain their existing definitions. The legacy fiscal correlation exercise
is kept as an appendix. Its lag counts are **observation rows after the joined sample
is filtered**, so missing macro quarters can make a row interval longer than one
calendar quarter. It is supplementary exploration and should not be described as a
precise calendar-quarter forecast or included as a dashboard stress channel.

In [ ]:
us_government_balance = get_fred_series("AD01RC1Q027SBEA", "us_government_balance")
us_gdp_nominal = get_fred_series("GDP", "us_nominal_gdp")
us_fiscal = safe_merge(us_government_balance.dropna(), us_gdp_nominal.dropna(), "date")
us_fiscal["quarter"] = us_fiscal["date"].dt.to_period("Q")
us_fiscal["us_fiscal_balance_pct_gdp"] = us_fiscal["us_government_balance"] / us_fiscal["us_nominal_gdp"] * 100
eurozone_fiscal = get_eurostat_series("gov_10q_ggnfa", {
    "format": "JSON", "lang": "en", "freq": "Q", "unit": "PC_GDP", "sector": "S13", "na_item": "B9", "s_adj": "SCA", "geo": "EA21"
}, "eurozone_fiscal_balance_pct_gdp", "eurostat_fiscal_ea21")
eurozone_fiscal["quarter"] = eurozone_fiscal["date"].dt.to_period("Q")
fiscal_comparison = safe_merge(us_fiscal[["quarter", "us_fiscal_balance_pct_gdp"]], eurozone_fiscal[["quarter", "eurozone_fiscal_balance_pct_gdp"]], "quarter")
fiscal_comparison = fiscal_comparison[fiscal_comparison["quarter"] <= QUARTER_END].copy()
for region in ["us", "eurozone"]:
    balance = fiscal_comparison[f"{region}_fiscal_balance_pct_gdp"]
    fiscal_comparison[f"{region}_fiscal_qoq_change"] = balance.diff()
    fiscal_comparison[f"{region}_fiscal_yoy_change"] = balance - balance.shift(4)
    fiscal_comparison[f"{region}_fiscal_direction"] = np.select([fiscal_comparison[f"{region}_fiscal_qoq_change"] < 0, fiscal_comparison[f"{region}_fiscal_qoq_change"] > 0], ["Deficit Widening", "Deficit Narrowing"], default="Unchanged")
fiscal_market = safe_merge(fiscal_comparison, market_confirmation[["quarter", "us_10y_yield", "us_10y_qoq_change", "eurozone_10y_yield", "eurozone_10y_qoq_change"]], "quarter")
lag_results = []
same_direction_rows = []
for region, prefix in [("US", "us"), ("Eurozone", "eurozone")]:
    fiscal_column, yield_column = f"{prefix}_fiscal_qoq_change", f"{prefix}_10y_qoq_change"
    sample = fiscal_market[["quarter", fiscal_column, yield_column]].dropna().copy()
    same_direction_rows.append({"region": region, "observations": len(sample), "same_direction_share": ((sample[fiscal_column] < 0) == (sample[yield_column] > 0)).mean()})
    for lag in range(5):
        lagged = sample.copy()
        lagged["future_yield_change"] = lagged[yield_column].shift(-lag)
        lagged = lagged.dropna()
        lag_results.append({"region": region, "lag_quarters": lag, "correlation": lagged[fiscal_column].corr(lagged["future_yield_change"]), "observations": len(lagged)})
fiscal_lag_analysis = pd.DataFrame(lag_results)
fiscal_comparison.to_csv(APPENDIX_DIR / "fiscal_comparison.csv", index=False)
fiscal_market.to_csv(APPENDIX_DIR / "fiscal_market.csv", index=False)
fiscal_lag_analysis.to_csv(APPENDIX_DIR / "fiscal_lag_analysis.csv", index=False)
pd.DataFrame(same_direction_rows).to_csv(APPENDIX_DIR / "fiscal_yield_same_direction.csv", index=False)
run_metadata["raw_sources_used"] = source_usage
(APPENDIX_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
display(fiscal_lag_analysis)
print("Notebook 01 complete. Production and supplementary outputs are saved separately.")
